In [1]:
%pip install gymnasium

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 952.1/952.1 kB 2.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [gymnasium]/3 [gymnasium]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade numpy==1.26.4
%pip uninstall tensorflow -y 
%pip install tensorflow==2.16.2 


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Found existing installation: tensorflow 2.16.2
Uninstalling tensorflow-2.16.2:
  Successfully uninstalled tensorflow-2.16.2
Note: you may need to restart the kernel to use updated packages.
  Using cached tensorflow-2.16.2-cp311-cp311-macosx_12_0_arm64.whl.metadata (4.1 kB)
Using cached tensorflow-2.16.2-cp311-cp311-macosx_12_0_arm64.whl (227.0 MB)

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import os 
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0' 
os.environ['CUDA_VISIBLE_DEVICES'] = '-1' 

In [4]:
import sys 
sys.setrecursionlimit(1500) 

import gymnasium as gym
import numpy as np 

# Create the environment 
env = gym.make('CartPole-v1') 

# Set random seed for reproducibility 
np.random.seed(42) 
env.action_space.seed(42) 
env.observation_space.seed(42)

42

Define the Q-Learning Model

In [6]:
# Suppress warnings for a cleaner notebook or console experience
import warnings
warnings.filterwarnings('ignore')

# Override the default warning function
def warn(*args, **kwargs):
    pass
warnings.warn = warn

# Import necessary libraries for the Q-Learning model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input  # Import Input layer
from tensorflow.keras.optimizers import Adam
import gymnasium as gym  # Ensure the environment library is available

# Define the model building function
def build_model(state_size, action_size): 
    model = Sequential() 
    model.add(Input(shape=(state_size,)))  # Use Input layer to specify the input shape 
    model.add(Dense(24, activation='relu')) 
    model.add(Dense(24, activation='relu')) 
    model.add(Dense(action_size, activation='linear')) 
    model.compile(loss='mse', optimizer=Adam(learning_rate=0.001)) 
    return model 

# Create the environment and set up the model
env = gym.make('CartPole-v1')
state_size = env.observation_space.shape[0] 
action_size = int(env.action_space.n)
model = build_model(state_size, action_size)


Implement the Q-Learning Algorithm

In [8]:
# ================================
# 1. IMPORTAÇÕES
# ================================

# Biblioteca padrão para geração de números aleatórios
import random

# Biblioteca NumPy para operações matemáticas e vetoriais
import numpy as np

# deque é uma fila eficiente (usada como memória de replay)
from collections import deque

# Importa o TensorFlow (usado indiretamente pelo modelo)
import tensorflow as tf


# ================================
# 2. PARÂMETROS DE EXPLORAÇÃO (EPSILON-GREEDY)
# ================================

# epsilon controla o quanto o agente explora ações aleatórias
# Começa em 1.0 = 100% exploração
epsilon = 1.0

# Valor mínimo de epsilon (nunca para de explorar totalmente)
epsilon_min = 0.01

# Fator de decaimento do epsilon
# A cada treino, epsilon diminui 1%
epsilon_decay = 0.99


# ================================
# 3. REPLAY MEMORY
# ================================

# Cria uma memória que guarda até 2000 experiências
# Quando enche, descarta automaticamente as mais antigas
memory = deque(maxlen=2000)


# ================================
# 4. FUNÇÃO PARA ARMAZENAR EXPERIÊNCIAS
# ================================

def remember(state, action, reward, next_state, done):
    """
    Armazena uma experiência na memória.
    
    Cada experiência contém:
    - state: estado atual
    - action: ação tomada
    - reward: recompensa recebida
    - next_state: próximo estado
    - done: se o episódio terminou
    """
    
    memory.append((state, action, reward, next_state, done))


# ================================
# 5. FUNÇÃO DE TREINAMENTO (REPLAY)
# ================================

def replay(batch_size=64):
    """
    Treina o modelo usando um conjunto aleatório de experiências da memória.
    """

    # Se não tiver experiências suficientes, não treina
    if len(memory) < batch_size:
        return

    # Seleciona aleatoriamente um conjunto de experiências
    minibatch = random.sample(memory, batch_size)

    # Separa cada elemento do minibatch em arrays
    states = np.vstack([x[0] for x in minibatch])
    actions = np.array([x[1] for x in minibatch])
    rewards = np.array([x[2] for x in minibatch])
    next_states = np.vstack([x[3] for x in minibatch])
    dones = np.array([x[4] for x in minibatch])

    # Prediz os valores Q para os próximos estados
    q_next = model.predict(next_states)

    # Prediz os valores Q para os estados atuais
    q_target = model.predict(states)

    # Atualiza os valores-alvo (Q-values)
    for i in range(batch_size):

        # Começa com a recompensa imediata
        target = rewards[i]

        # Se o episódio não terminou,
        # soma a recompensa futura descontada (fator gamma = 0.95)
        if not dones[i]:
            target += 0.95 * np.amax(q_next[i])

        # Atualiza apenas o Q-value da ação tomada
        q_target[i][actions[i]] = target

    # Treina o modelo com os valores Q atualizados
    model.fit(states, q_target, epochs=1, verbose=0)

    # ================================
    # 6. ATUALIZAÇÃO DO EPSILON
    # ================================

    # Usa a variável global epsilon
    global epsilon

    # Reduz epsilon gradualmente
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay


# ================================
# 7. FUNÇÃO PARA ESCOLHER A AÇÃO
# ================================

def act(state):
    """
    Escolhe uma ação usando a política epsilon-greedy.
    """

    # Exploração: ação aleatória
    if np.random.rand() <= epsilon:
        return random.randrange(action_size)

    # Exploração zero: rede neural decide
    act_values = model.predict(state)

    # Escolhe a ação com maior valor Q
    return np.argmax(act_values[0])


# ================================
# 8. PARÂMETROS DE TREINAMENTO
# ================================

# Número total de episódios
episodes = 10

# Frequência de treinamento (treina a cada 5 passos)
train_frequency = 5


# ================================
# 9. LOOP PRINCIPAL DE TREINAMENTO
# ================================

for e in range(episodes):

    # Reinicia o ambiente
    # Gymnasium retorna (estado, info)
    state, _ = env.reset()

    # Ajusta o formato do estado para a rede neural
    state = np.reshape(state, [1, state_size])

    # Cada episódio roda até 200 passos no máximo
    for time in range(200):

        # Escolhe uma ação
        action = act(state)

        # Executa a ação no ambiente
        next_state, reward, terminated, truncated, _ = env.step(action)

        # Define se o episódio acabou
        done = terminated or truncated

        # Penaliza se o episódio terminou
        reward = reward if not done else -10

        # Ajusta o formato do próximo estado
        next_state = np.reshape(next_state, [1, state_size])

        # Armazena a experiência
        remember(state, action, reward, next_state, done)

        # Atualiza o estado atual
        state = next_state

        # Se terminou o episódio, exibe resultado
        if done:
            print(f"episode: {e+1}/{episodes}, score: {time}, e: {epsilon:.2}")
            break

        # Treina o modelo a cada "train_frequency" passos
        if time % train_frequency == 0:
            replay(batch_size=64)


# ================================
# 10. FINALIZAÇÃO
# ================================

# Fecha o ambiente corretamente
env.close()

episode: 1/10, score: 41, e: 1.0
episode: 2/10, score: 10, e: 1.0
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
episode: 3/10, score: 15, e: 0.99
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
episode: 4/10, score: 16, e: 0.95
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step
episode: 5/10, score: 13, e: 0.92
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step
2/2 ━━━━━━━

Evaluate the Performance

In [9]:
for e in range(10):  

    state, _ = env.reset()  # Unpack the state from the tuple 
    state = np.reshape(state, [1, state_size])  # Reshape the state correctly 
    for time in range(500):  
        env.render()  
        action = np.argmax(model.predict(state)[0])  
        next_state, reward, terminated, truncated, _ = env.step(action)  # Unpack the five return values 
        done = terminated or truncated  # Check if the episode is done 
        next_state = np.reshape(next_state, [1, state_size])  
        state = next_state  
        if done:  
            print(f"episode: {e+1}/10, score: {time}")  
            break  

env.close() 

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
episode: 1/10, score: 8
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step
episode: 2/10, score: 9
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step
1/